# R Master v7a · Body Surface Rebalance · 一键断点版

这一版从 v6 的 **True Body Shell** 继续，目标是修正“骨架已经窄了，但髋/臀/大腿根表面仍然偏 Mona 原始体块”的问题。

### 只做表面，不再改骨长
- 保留 v2 已通过的四肢比例与髋关节宽度；
- 保留 505 骨结构；
- 不烘焙最终 Rest Pose；
- 不导出最终 VRM；
- 不改 Drive 里的 v2 源文件。

### v7a 是保守第一轮
只对 `R2_Mona_Main` 做平滑区域变形：
- 髋部外侧：温和收窄；
- 腰 → 骨盆：平滑过渡；
- 大腿根：避免跟着髋部一起被过度夹紧；
- 臀部后侧：轻微减小后突体块；
- 中央前侧成人结构区域不做专门编辑。

### 输出 8 张
1. 全身正面
2. 全身侧面
3. 全身背面
4. 全身 3/4
5. 胸廓 → 腰
6. 腰 → 骨盆
7. 骨盆 → 大腿根
8. 臀线侧视

每张图完成后立即写入 Google Drive。中途断线后重新 **运行全部**，已经完成的阶段会自动跳过。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, json, os, math

BUILD_TAG="v7a_surface_20260919_r1"

print("R Master v7a · Body Surface Rebalance")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v2"/"latest"/"R_Master_Align_v2_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v7a_surface"/"latest"

CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size < 50*1024*1024:
    raise RuntimeError("没找到 v2 预览文件。把这一屏截图给二蛋即可。")

print(f"✓ v2 源：{SRC.stat().st_size/1024/1024:.1f} MiB")
print("✓ 输出目录：MyDrive/R_Master/v7a_surface/latest/")



In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"

LOCAL=Path("/content/r_master_v7a")
LOCAL.mkdir(parents=True,exist_ok=True)

ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(
        ["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],
        check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT
    )

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size > 100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender Drive 缓存")
else:
    print("补下载 Blender 一次…")
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)
    print("✓ Blender 已写入 Drive 缓存")

if not (BDIR/"blender").exists():
    if BDIR.exists():
        shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)

BLENDER=BDIR/"blender"
if not BLENDER.exists():
    raise RuntimeError("Blender 解压失败。")

print("✓ Blender 4.4.3 就绪")



In [ ]:
BUILD_SCRIPT=LOCAL/"R_Master_v7a_Build.py"
RENDER_SCRIPT=LOCAL/"R_Master_v7a_Render.py"
BUILD_SCRIPT.write_text("\nimport bpy, os, sys, json, math\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None\ntag=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--tag\" and i+1<len(argv): tag=argv[i+1]\nif not out: raise RuntimeError(\"missing --out\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\nif not body or body.type!=\"MESH\":\n    raise RuntimeError(\"R2_Mona_Main missing\")\nif not rig or rig.type!=\"ARMATURE\":\n    raise RuntimeError(\"R Master/Mona armature missing\")\n\n# True body shell isolation, same principle as v6.\nremoved_meshes=[]\nfor obj in list(bpy.data.objects):\n    if obj.type==\"MESH\" and obj != body:\n        removed_meshes.append(obj.name)\n        bpy.data.objects.remove(obj,do_unlink=True)\n\ndisabled=[]\nfor mod in body.modifiers:\n    if (\n        mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\",\"PARTICLE_SYSTEM\"}\n        or mod.name in {\"Mask\",\"Mask.001\"}\n        or \"mask\" in mod.name.lower()\n    ):\n        disabled.append({\"name\":mod.name,\"type\":mod.type})\n        mod.show_viewport=False\n        mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1)\n        mod.render_levels=min(mod.render_levels,1)\n\nbody.hide_set(False)\nbody.hide_viewport=False\nbody.hide_render=False\nbpy.context.view_layer.update()\n\nmesh=body.data\nmw=body.matrix_world.copy()\nimw=mw.inverted()\n\n# Body base-mesh world-space bounds.\norig_world=[mw @ v.co for v in mesh.vertices]\nmn=Vector((min(p.x for p in orig_world),min(p.y for p in orig_world),min(p.z for p in orig_world)))\nmx=Vector((max(p.x for p in orig_world),max(p.y for p in orig_world),max(p.z for p in orig_world)))\ncenter=(mn+mx)*0.5\nheight=max(1e-6,mx.z-mn.z)\nhalf_width=max(1e-6,(mx.x-mn.x)*0.5)\ndepth=max(1e-6,mx.y-mn.y)\n\n# Determine which Y direction is posterior from Mona's own anatomy-control bones.\nposterior_sign=1.0\ntry:\n    ass=rig.pose.bones.get(\"Assbutt.L\") or rig.pose.bones.get(\"Ass.L\")\n    front=rig.pose.bones.get(\"Labia.L\") or rig.pose.bones.get(\"LowerBodyStretch\")\n    if ass and front:\n        ay=(rig.matrix_world @ ass.head).y\n        fy=(rig.matrix_world @ front.head).y\n        posterior_sign=1.0 if ay>=fy else -1.0\nexcept Exception:\n    posterior_sign=1.0\n\ndef smoothstep(a,b,x):\n    if a==b:\n        return 1.0 if x>=b else 0.0\n    t=max(0.0,min(1.0,(x-a)/(b-a)))\n    return t*t*(3.0-2.0*t)\n\ndef bell(x,a,b,c,d):\n    # 0 -> smooth rise -> 1 plateau -> smooth fall -> 0\n    return smoothstep(a,b,x) * (1.0-smoothstep(c,d,x))\n\nmoved=0\nsum_disp=0.0\nmax_disp=0.0\n\n# Conservative first pass. All edits are continuous falloff fields.\nfor i,v in enumerate(mesh.vertices):\n    p=orig_world[i].copy()\n    q=p.copy()\n    t=(p.z-mn.z)/height\n    xrel=p.x-center.x\n    yrel=p.y-center.y\n    ax=abs(xrel)\n\n    # A) Hip lateral soft-tissue rebalance.\n    # Strongest around mid-pelvis, tapered before waist and upper-thigh.\n    hip_band=bell(t,0.405,0.445,0.535,0.575)\n    lateral=smoothstep(0.055*half_width,0.42*half_width,ax)\n    # Up to 9.5% X reduction only at the outer hip.\n    hip_strength=0.095*hip_band*lateral\n    q.x=center.x + xrel*(1.0-hip_strength)\n\n    # B) Lower-waist continuity: tiny outward support to avoid an hourglass kink.\n    waist_band=bell(t,0.535,0.565,0.605,0.635)\n    waist_lateral=smoothstep(0.04*half_width,0.32*half_width,ax)\n    waist_support=0.018*waist_band*waist_lateral\n    q.x += xrel*waist_support\n\n    # C) Upper-thigh transition protection.\n    # Mildly restore lateral volume below the hip so the root does not pinch.\n    thigh_band=bell(t,0.345,0.375,0.425,0.455)\n    thigh_lateral=smoothstep(0.05*half_width,0.38*half_width,ax)\n    thigh_support=0.015*thigh_band*thigh_lateral\n    q.x += xrel*thigh_support\n\n    # D) Posterior glute projection: mild reduction only on the posterior side.\n    post=(posterior_sign*yrel)\n    post_mask=smoothstep(0.08*depth,0.24*depth,post)\n    glute_band=bell(t,0.39,0.425,0.515,0.555)\n    glute_strength=0.055*glute_band*post_mask\n    # move posterior vertices toward body's Y center; anterior side is untouched\n    q.y=center.y + yrel*(1.0-glute_strength)\n\n    # World-space single-vertex safety cap: 22 mm.\n    delta=q-p\n    d=delta.length\n    if d>0.022:\n        delta*=0.022/d\n        q=p+delta\n        d=0.022\n\n    if d>1e-6:\n        moved+=1\n        sum_disp+=d\n        max_disp=max(max_disp,d)\n        v.co=imw @ q\n\nbpy.context.view_layer.update()\n\n# New base-mesh bbox after rebalance.\nnew_world=[mw @ v.co for v in mesh.vertices]\nmn2=Vector((min(p.x for p in new_world),min(p.y for p in new_world),min(p.z for p in new_world)))\nmx2=Vector((max(p.x for p in new_world),max(p.y for p in new_world),max(p.z for p in new_world)))\n\nbody[\"red_master_stage\"]=\"R_Master_SurfaceRebalance_v7a\"\nbody[\"red_master_build_tag\"]=tag or \"v7a\"\nbody[\"red_master_restpose_baked\"]=False\nbody[\"red_master_surface_rebalanced\"]=True\n\nreport={\n    \"ok\":True,\n    \"stage\":\"R_Master_SurfaceRebalance_v7a\",\n    \"build_tag\":tag,\n    \"source_blend\":bpy.data.filepath,\n    \"body_object\":body.name,\n    \"rig_object\":rig.name,\n    \"bone_count\":len(rig.data.bones),\n    \"remaining_mesh_objects\":[o.name for o in bpy.data.objects if o.type==\"MESH\"],\n    \"removed_mesh_count\":len(removed_meshes),\n    \"removed_mesh_objects\":removed_meshes,\n    \"disabled_modifiers\":disabled,\n    \"posterior_sign\":posterior_sign,\n    \"surface_edit\":{\n        \"moved_vertex_count\":moved,\n        \"total_vertex_count\":len(mesh.vertices),\n        \"mean_displacement_m\":(sum_disp/moved if moved else 0.0),\n        \"max_displacement_m\":max_disp,\n        \"safety_cap_m\":0.022,\n        \"hip_max_lateral_reduction\":0.095,\n        \"waist_support_max\":0.018,\n        \"upper_thigh_support_max\":0.015,\n        \"posterior_glute_reduction_max\":0.055\n    },\n    \"bbox_before\":{\"min\":list(map(float,mn)),\"max\":list(map(float,mx))},\n    \"bbox_after\":{\"min\":list(map(float,mn2)),\"max\":list(map(float,mx2))},\n    \"rest_pose_baked\":False,\n    \"final_vrm\":False,\n    \"notes\":[\n        \"Conservative v7a surface-only rebalance.\",\n        \"No bone lengths changed in this stage.\",\n        \"No final rest-pose bake.\",\n        \"No final VRM export.\",\n        \"Adult-structure bones were not deleted or renamed.\"\n    ]\n}\n\nreport_path=os.path.join(out,\"R_Master_v7a_report.json\")\nwith open(report_path,\"w\",encoding=\"utf-8\") as f:\n    json.dump(report,f,ensure_ascii=False,indent=2)\n\nblend_path=os.path.join(out,\"R_Master_v7a_SURFACE_PREVIEW.blend\")\nbpy.ops.wm.save_as_mainfile(filepath=blend_path,check_existing=False)\n\nprint(\"[R Master v7a] BUILD_OK\")\nprint(\"[R Master v7a] moved:\",moved,\"/\",len(mesh.vertices))\nprint(\"[R Master v7a] mean/max displacement:\",report[\"surface_edit\"][\"mean_displacement_m\"],max_disp)\nprint(\"[R Master v7a] posterior_sign:\",posterior_sign)\nprint(\"[R Master v7a] blend:\",blend_path)\n",encoding="utf-8")
RENDER_SCRIPT.write_text("\nimport bpy, os, sys, json\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None; view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--view\" and i+1<len(argv): view=argv[i+1]\nif not out or not view: raise RuntimeError(\"missing --out/--view\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nif not body: raise RuntimeError(\"R2_Mona_Main missing\")\n\nfor obj in bpy.context.scene.objects:\n    if obj.type==\"MESH\":\n        obj.hide_render=(obj!=body)\n        obj.hide_viewport=(obj!=body)\n    elif obj.type==\"ARMATURE\":\n        obj.hide_render=True\n\nfor mod in body.modifiers:\n    if mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\",\"PARTICLE_SYSTEM\"} or \"mask\" in mod.name.lower():\n        mod.show_viewport=False\n        mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1)\n        mod.render_levels=min(mod.render_levels,1)\n\nbody.hide_render=False\nbody.hide_viewport=False\nbpy.context.view_layer.update()\n\npts=[body.matrix_world @ Vector(c) for c in body.bound_box]\nmn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\nmx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\ncenter=(mn+mx)*0.5\nheight=mx.z-mn.z; width=mx.x-mn.x; depth=mx.y-mn.y\ndist=max(height,width,depth)*2.5\n\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_WORKBENCH\"\nscene.render.image_settings.file_format=\"PNG\"\nscene.render.film_transparent=False\nscene.display.shading.light=\"STUDIO\"\nscene.display.shading.show_shadows=True\nscene.display.shading.show_cavity=True\nscene.display.shading.cavity_type=\"WORLD\"\nscene.display.shading.color_type=\"SINGLE\"\nscene.display.shading.single_color=(0.60,0.60,0.63)\nscene.display.shading.background_type=\"VIEWPORT\"\nscene.display.shading.background_color=(0.04,0.04,0.05)\n\ncam_data=bpy.data.cameras.get(\"R_Master_v7a_Camera_DATA\") or bpy.data.cameras.new(\"R_Master_v7a_Camera_DATA\")\ncam=bpy.data.objects.get(\"R_Master_v7a_Camera\")\nif not cam:\n    cam=bpy.data.objects.new(\"R_Master_v7a_Camera\",cam_data)\n    scene.collection.objects.link(cam)\nscene.camera=cam\ncam.data.type=\"ORTHO\"\n\ndef look_at(obj,target):\n    obj.rotation_euler=(Vector(target)-obj.location).to_track_quat(\"-Z\",\"Y\").to_euler()\n\ndef render_file(fname,pos,target,scale,res):\n    scene.render.resolution_x,scene.render.resolution_y=res\n    scene.render.resolution_percentage=100\n    cam.location=Vector(pos)\n    cam.data.ortho_scale=scale\n    look_at(cam,Vector(target))\n    path=os.path.join(out,fname)\n    scene.render.filepath=path\n    bpy.ops.render.render(write_still=True)\n    return path\n\nspec={\n \"full_front\":(\"R_Master_v7a_full_front.png\",(center.x,center.y-dist,center.z),center,height*1.08,(720,960)),\n \"full_side\":(\"R_Master_v7a_full_side.png\",(center.x+dist,center.y,center.z),center,height*1.08,(720,960)),\n \"full_back\":(\"R_Master_v7a_full_back.png\",(center.x,center.y+dist,center.z),center,height*1.08,(720,960)),\n \"full_three_quarter\":(\"R_Master_v7a_full_three_quarter.png\",(center.x+dist*.72,center.y-dist*.72,center.z),center,height*1.08,(720,960)),\n}\n\nfor key,frac in [(\"torso_waist\",.61),(\"waist_pelvis\",.51),(\"pelvis_upperthigh\",.42)]:\n    z=mn.z+height*frac\n    spec[key]=(f\"R_Master_v7a_{key}.png\",(center.x,center.y-dist,z),(center.x,center.y,z),max(.42,height*.25),(900,700))\n\n# Better glute-side framing than v6: slightly higher and wider.\nz=mn.z+height*.485\nspec[\"glute_side\"]=(\"R_Master_v7a_glute_side.png\",(center.x+dist,center.y,z),(center.x,center.y,z),max(.50,height*.30),(900,700))\n\nif view not in spec: raise RuntimeError(\"unknown view \"+str(view))\nfname,pos,target,scale,res=spec[view]\npath=render_file(fname,pos,target,scale,res)\nprint(\"[R Master v7a] RENDER_OK\",view,path)\n",encoding="utf-8")

STAGE_BLEND=OUT/"R_Master_v7a_SURFACE_PREVIEW.blend"
REPORT=OUT/"R_Master_v7a_report.json"
BUILD_LOG=OUT/"R_Master_v7a_build.log"

rebuild=True
if STAGE_BLEND.exists() and STAGE_BLEND.stat().st_size>50*1024*1024 and REPORT.exists():
    try:
        r=json.loads(REPORT.read_text(encoding="utf-8"))
        rebuild=(r.get("build_tag")!=BUILD_TAG)
    except:
        rebuild=True

if rebuild:
    print("③ 构建 v7a Surface Rebalance…")
    # Build locally first, then copy stage/report to Drive.
    LOCAL_STAGE=LOCAL/"stage"
    if LOCAL_STAGE.exists(): shutil.rmtree(LOCAL_STAGE)
    LOCAL_STAGE.mkdir(parents=True,exist_ok=True)
    local_log=LOCAL_STAGE/"R_Master_v7a_build.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(BUILD_SCRIPT),"--","--out",str(LOCAL_STAGE),"--tag",BUILD_TAG]
    with local_log.open("w",encoding="utf-8") as log:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "R Master v7a" in line or "Traceback" in line or "Error" in line:
                print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(local_log.read_text(encoding="utf-8",errors="replace")[-10000:])
        raise RuntimeError(f"v7a 构建失败，退出码 {rc}。截图给二蛋即可。")

    for name in ["R_Master_v7a_SURFACE_PREVIEW.blend","R_Master_v7a_report.json","R_Master_v7a_build.log"]:
        src=LOCAL_STAGE/name
        if not src.exists(): raise RuntimeError("v7a 构建缺少："+name)
        shutil.copy2(src,OUT/name)
    print("✓ v7a 重平衡结果已写入 Drive")
else:
    print("✓ v7a Surface Preview 已存在，跳过重建")

report=json.loads(REPORT.read_text(encoding="utf-8"))
print("  移动顶点：",report["surface_edit"]["moved_vertex_count"],"/",report["surface_edit"]["total_vertex_count"])
print("  平均位移：",round(report["surface_edit"]["mean_displacement_m"]*1000,2),"mm")
print("  最大位移：",round(report["surface_edit"]["max_displacement_m"]*1000,2),"mm")
print("  骨骼数：",report["bone_count"])
print("  Rest Pose 烘焙：",report["rest_pose_baked"])



In [ ]:
views=[
 ("full_front","R_Master_v7a_full_front.png"),
 ("full_side","R_Master_v7a_full_side.png"),
 ("full_back","R_Master_v7a_full_back.png"),
 ("full_three_quarter","R_Master_v7a_full_three_quarter.png"),
 ("torso_waist","R_Master_v7a_torso_waist.png"),
 ("waist_pelvis","R_Master_v7a_waist_pelvis.png"),
 ("pelvis_upperthigh","R_Master_v7a_pelvis_upperthigh.png"),
 ("glute_side","R_Master_v7a_glute_side.png"),
]

print("④ 断点渲染 8 视图…")
for idx,(view,fname) in enumerate(views,1):
    dest=OUT/fname
    if dest.exists() and dest.stat().st_size>20_000:
        print(f"✓ [{idx}/8] {view} 已存在，跳过")
        continue

    print(f"▶ [{idx}/8] {view}…")
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(STAGE_BLEND),"--python",str(RENDER_SCRIPT),"--","--out",str(OUT),"--view",view]
    p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    important=[x for x in p.stdout.splitlines() if "R Master v7a" in x or "Traceback" in x or "Error" in x]
    if important: print("\n".join(important[-10:]))
    if p.returncode!=0:
        raise RuntimeError(f"{view} 渲染失败，退出码 {p.returncode}。截图给二蛋即可。")
    if not dest.exists():
        raise RuntimeError(f"{view} 未生成。")
    print(f"✓ [{idx}/8] 已写入 Drive")

print("✓ v7a 八张图全部完成")



In [ ]:
from IPython.display import display,Image,Markdown

views=[
 ("全身正面","R_Master_v7a_full_front.png"),
 ("全身侧面","R_Master_v7a_full_side.png"),
 ("全身背面","R_Master_v7a_full_back.png"),
 ("全身 3/4","R_Master_v7a_full_three_quarter.png"),
 ("胸廓 → 腰","R_Master_v7a_torso_waist.png"),
 ("腰 → 骨盆","R_Master_v7a_waist_pelvis.png"),
 ("骨盆 → 大腿根","R_Master_v7a_pelvis_upperthigh.png"),
 ("臀线侧视","R_Master_v7a_glute_side.png"),
]

for title,fname in views:
    p=OUT/fname
    if p.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(p),width=500))

review=OUT/"R_Master_v7a_Review.zip"
if review.exists(): review.unlink()

with zipfile.ZipFile(review,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for _,fname in views:
        p=OUT/fname
        if p.exists(): z.write(p,arcname=p.name)
    for name in ["R_Master_v7a_report.json","R_Master_v7a_build.log"]:
        p=OUT/name
        if p.exists(): z.write(p,arcname=p.name)

print(f"✓ Review ZIP：{review.stat().st_size/1024/1024:.1f} MiB")
print("重 .blend 留在 Drive，不下载到手机。")
files.download(str(review))

